[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_9/00_tag_9_datensatz_vorbereiten.ipynb)

# Tag 9 – 00: Realen Datensatz für Objekterkennung herunterladen

Dieses Notebook lädt den **Penn-Fudan Pedestrian Dataset** herunter: echte
Straßen- und Campus-Szenen mit präzisen Personenmasken. Aus den Masken werden
Bounding Boxes abgeleitet. Der Download ist mit rund 51 MB bewusst klein genug
für einen Kurs; es gibt nur 170 Bilder und 345 annotierte Personen.


## Warum dieser Datensatz?

- **realistische Fotos** statt künstlich gezeichneter Formen,
- eine klar abgegrenzte Klasse: `Person`,
- pixelgenaue Masken, aus denen Bounding Boxes zuverlässig entstehen,
- klein genug für einen schnellen, einmaligen Download.

Das Notebook 01 bleibt bewusst einfach und verwendet nur Bilder mit **genau
einer** annotierten Person. Das ist keine Datensatzerzeugung: Es ist eine
reproduzierbare Auswahl aus den originalen, heruntergeladenen Annotationen.
Mehrpersonenbilder bleiben unverändert im Download-Ordner und sind eine gute
spätere Erweiterung.


In [ ]:
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from torchvision.datasets.utils import download_and_extract_archive

RANDOM_STATE = 42
DATA_DIR = Path("datasets")
DATASET_ROOT = DATA_DIR / "PennFudanPed"
DOWNLOAD_URL = "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip"
ARCHIVE_NAME = "PennFudanPed.zip"

print("Datensatzordner:", DATASET_ROOT.resolve())


## Einmalig herunterladen und entpacken

Wird der Ordner bereits gefunden, startet kein weiterer Download. In einer
normalen Internetverbindung ist der rund 51-MB-Download schnell abgeschlossen.


In [ ]:
image_directory = DATASET_ROOT / "PNGImages"
mask_directory = DATASET_ROOT / "PedMasks"

if image_directory.exists() and mask_directory.exists():
    print("Penn-Fudan ist bereits entpackt – Download wird übersprungen.")
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    download_and_extract_archive(
        url=DOWNLOAD_URL,
        download_root=str(DATA_DIR),
        extract_root=str(DATA_DIR),
        filename=ARCHIVE_NAME,
    )
    print("Download und Entpacken abgeschlossen.")

print("Bilder:", len(list(image_directory.glob("*.png"))))
print("Masken:", len(list(mask_directory.glob("*.png"))))


## Originalmasken in Bounding-Box-Labels umwandeln

Die Masken enthalten pro Person eine eigene ID. Für jede ID bestimmen wir die
äußersten Pixel und erhalten `[x_min, y_min, x_max, y_max]`. Für das einfache
Anker-Beispiel wählen wir ausschließlich Bilder mit einer Person aus und teilen
diese deterministisch in Training und Validierung auf.


In [ ]:
def boxes_from_mask(mask_path):
    mask = np.asarray(Image.open(mask_path))
    person_ids = np.unique(mask)
    person_ids = person_ids[person_ids != 0]  # 0 ist Hintergrund
    boxes = []
    for person_id in person_ids:
        y_coordinates, x_coordinates = np.where(mask == person_id)
        boxes.append([
            int(x_coordinates.min()), int(y_coordinates.min()),
            int(x_coordinates.max()) + 1, int(y_coordinates.max()) + 1,
        ])
    return boxes


annotations = []
for image_path in sorted(image_directory.glob("*.png")):
    mask_path = mask_directory / f"{image_path.stem}_mask.png"
    boxes = boxes_from_mask(mask_path)
    if len(boxes) == 1:  # einfacher Ein-Objekt-Fall für Notebook 01
        annotations.append({
            "image": image_path.relative_to(DATASET_ROOT).as_posix(),
            "box_xyxy": boxes[0],
            "label": 1,
        })

rng = random.Random(RANDOM_STATE)
rng.shuffle(annotations)
split_index = int(0.8 * len(annotations))
train_annotations = annotations[:split_index]
valid_annotations = annotations[split_index:]

(DATASET_ROOT / "train_annotations.json").write_text(
    json.dumps(train_annotations, indent=2), encoding="utf-8"
)
(DATASET_ROOT / "valid_annotations.json").write_text(
    json.dumps(valid_annotations, indent=2), encoding="utf-8"
)

print(f"Ein-Person-Bilder: {len(annotations)} von 170")
print(f"Training: {len(train_annotations)}, Validierung: {len(valid_annotations)}")


## Bilder und echte Labels prüfen

Die gelbe Umrandung stammt aus der Originalmaske. Sie wird nicht in die
Bilddatei geschrieben und ist daher kein Teil des Modell-Inputs.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, annotation in zip(axes.flat, train_annotations[:6]):
    image = np.asarray(Image.open(DATASET_ROOT / annotation["image"]).convert("RGB"))
    x0, y0, x1, y1 = annotation["box_xyxy"]
    ax.imshow(image)
    ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                               fill=False, edgecolor="yellow", linewidth=2))
    ax.set_title(f"Person: {annotation['box_xyxy']}")
    ax.axis("off")
plt.tight_layout()


In [ ]:
assert train_annotations and valid_annotations
first = train_annotations[0]
assert (DATASET_ROOT / first["image"]).exists()
assert len(first["box_xyxy"]) == 4

print("OK – reales Beispielbild:", first["image"])
print("OK – echte Bounding Box [x_min, y_min, x_max, y_max]:", first["box_xyxy"])
